In [1]:
import re
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from datetime import datetime
from plotnine import ggplot, geom_bar, aes, geom_col, labs, theme, element_blank, element_text

In [2]:
# Part one of this script adds a "priority" column to metadata

In [3]:
# This will allow me to run --priorities in augur filter

In [4]:
# Step 1: read in metadata

METADATA_pb2 = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb2_detectionsgeodata.tsv"
METADATA_pb1 = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb1_detectionsgeodata.tsv"
METADATA_pa = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_pa_detectionsgeodata.tsv"
METADATA_ha = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha_detectionsgeodata.tsv"
METADATA_np = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_np_detectionsgeodata.tsv"
METADATA_na = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_na_detectionsgeodata.tsv"
METADATA_mp = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_m_detectionsgeodata.tsv"
METADATA_ns = "./metadata/sequence_metadata/NCBI-BVBRC-GIS_ns_detectionsgeodata.tsv"

In [5]:
pb2_metadata = pd.read_csv(METADATA_pb2, sep = '\t')
pb1_metadata = pd.read_csv(METADATA_pb1, sep = '\t')
pa_metadata = pd.read_csv(METADATA_pa, sep = '\t')
ha_metadata = pd.read_csv(METADATA_ha, sep = '\t')
np_metadata = pd.read_csv(METADATA_np, sep = '\t')
na_metadata = pd.read_csv(METADATA_na, sep = '\t')
mp_metadata = pd.read_csv(METADATA_mp, sep = '\t')
ns_metadata = pd.read_csv(METADATA_ns, sep = '\t')

In [6]:
# Function: add priority based on lat_long_source

def priorities(metadata):
    priority_dict = {"county": "1", "city": "2", "state": "3"}
    
    # Use map and fill NaN with a default priority if desired
    metadata["priority"] = metadata["lat_long_source"].map(priority_dict).fillna("4")
    
    return metadata

In [7]:
# Step 2: Add column using the above function

pb2_metadata_prioritized = priorities(pb2_metadata)
pb1_metadata_prioritized = priorities(pb2_metadata)
pa_metadata_prioritized = priorities(pb2_metadata)
ha_metadata_prioritized = priorities(pb2_metadata)
np_metadata_prioritized = priorities(pb2_metadata)
na_metadata_prioritized = priorities(pb2_metadata)
mp_metadata_prioritized = priorities(pb2_metadata)
ns_metadata_prioritized = priorities(pb2_metadata)

In [8]:
print(len(pb2_metadata_prioritized))
print(pb2_metadata_prioritized["priority"].value_counts())

6608
priority
1    3991
3    2596
2      21
Name: count, dtype: int64


In [9]:
# Step 3: subset to just the required columns

pb2_prioritized_strains = pb2_metadata_prioritized.loc[0:, ["strain", "priority"]]
pb1_prioritized_strains = pb1_metadata_prioritized.loc[0:, ["strain", "priority"]]
pa_prioritized_strains = pa_metadata_prioritized.loc[0:, ["strain", "priority"]]
ha_prioritized_strains = ha_metadata_prioritized.loc[0:, ["strain", "priority"]]
np_prioritized_strains = np_metadata_prioritized.loc[0:, ["strain", "priority"]]
na_prioritized_strains = na_metadata_prioritized.loc[0:, ["strain", "priority"]]
mp_prioritized_strains = mp_metadata_prioritized.loc[0:, ["strain", "priority"]]
ns_prioritized_strains = ns_metadata_prioritized.loc[0:, ["strain", "priority"]]

In [10]:
# Step 4: write out TSV

pb2_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_pb2.tsv", sep = '\t', header = None, index = False)
pb1_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_pb1.tsv", sep = '\t', header = None, index = False)
pa_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_pa.tsv", sep = '\t', header = None, index = False)
ha_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_ha.tsv", sep = '\t', header = None, index = False)
np_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_np.tsv", sep = '\t', header = None, index = False)
na_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_na.tsv", sep = '\t', header = None, index = False)
mp_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_m.tsv", sep = '\t', header = None, index = False)
ns_prioritized_strains.to_csv("./metadata/sequence_metadata/list_priorities_ns.tsv", sep = '\t', header = None, index = False)

In [ ]:
# Part two adds a column for latitude_group OR makes a smaller dataframe of just the strain name and latitude_group

In [ ]:
# This allows me to use latitude_group as a trait in a DTA

In [32]:
da_data = pd.read_csv("./BEAST/discrete_trait_analysis/submission_files/2025-09-01_flyway-latitude-order_emptree-contphyl/inputs/div-tree_metadata.tsv", sep = '\t')

In [33]:
FLY = "./metadata/helpful_lists/flywayregions_biggerlist.csv"

flyways = pd.read_csv(FLY, sep = ",")

In [34]:
# Make a dictionary of the flyways dataframe where each entry in the "location" column is a key
flyways_dict = flyways.set_index('location').to_dict(orient='index')

# Make a smaller dictionary of just latitude_group
latitude_map = {loc: data['latitude_group'] for loc, data in flyways_dict.items()}

In [35]:
# Add a latitude_group column

ha_metadata_withlatitude = da_data.copy()

ha_metadata_withlatitude["latitude_group"] = ha_metadata_withlatitude["division"].map(latitude_map)

In [36]:
# This code chunck changes the "southern_mexico" and "northern_mexico" labels to "central_america" because I simply have to sequences from wild birds in mexico

ha_metadata_withlatitude.loc[ha_metadata_withlatitude["latitude_group"].isin(["southern_mexico", "northern_mexico"]), "latitude_group"] = "central_america"

In [37]:
# Optional: keep just the strain and latitude_gorup columns

keep_columns = ["strain", "latitude_group"]

limit = ha_metadata_withlatitude[keep_columns].rename({"strain":"taxa", "latitude_group":"lat_group"}, axis = 1)

In [38]:
limit.to_csv("./BEAST/discrete_trait_analysis/submission_files/2025-09-01_flyway-latitude-order_emptree-contphyl/inputs/div-tree_lat-group.txt", sep = '\t')

In [3]:
# Part three of this script takes a list of strains and adds a column for the host order

In [4]:
# This allows me to add the host order as a trait on a DTA using a separate CSV

In [3]:
# Upload the list of strains

# PATH = "./BEAST/discrete_trait_analysis/submission_files/2025-08-05_order_emptree-contphyl/inputs/tree_strains.txt"
PATH = "./BEAST/continuous_phylogeography/submission_files/2025-08-11_nov2021-2024_no-suliformes_ha_contphyl_flyway-order-DTA_thorney/inputs/div-tree_strains.txt"

with open(PATH, "r") as STRAINS:
    strain_list = STRAINS.read().splitlines()

# Make this into a dataframe

strain_list_df = pd.DataFrame(strain_list, columns = ["taxa"])

In [4]:
# Add a column for original host order

strain_list_df["original_order"] = strain_list_df["taxa"].str.split("|", expand = True)[3]

print(set(strain_list_df["original_order"]))

{'passeriformes', 'accipitriformes', 'strigiformes', 'ciconiiformes', 'gaviiformes', 'unknown', 'columbiformes', 'casuariiformes', 'gruiformes', 'apodiformes', 'podicipediformes', 'suliformes', 'pelecaniformes', 'falconiformes', 'charadriiformes', 'anseriformes', 'galliformes'}


In [5]:
# The above is a come correct and complete list of host orders
# However, 17 orders is far too many traits for a DTA
# Going to correct the host orders so I have fewer orders

host_order_corrections_dict = {"accipitriformes":"accipitriformes/strigiformes", # eagles, hawks, and most raptors
                               "anseriformes":"anseriformes", # duckies
                               "apodiformes":"other", # swifts and hummingbirds (a.k.a. cutie patooties)
                               "casuariiformes":"other", # emus, cassowaries
                               "charadriiformes":"charadriiformes/shorebirds", # gulls and terns
                               "ciconiiformes":"charadriiformes/shorebirds", # storks
                               "columbiformes":"other", # doves, pigeons
                               "falconiformes":"other", # falcons
                               "galliformes":"other", # ground birds
                               "gaviiformes":"other", # loons, divers
                               "gruiformes":"charadriiformes/shorebirds", # cranes and other waders
                               "passeriformes":"other", # songbirds and corvids
                               "pelecaniformes":"charadriiformes/shorebirds", # pelicans, obvi (possibly polyphyletic)
                               "podicipediformes":"other", # grebes
                               "strigiformes":"accipitriformes/strigiformes", # owlies
                               "suliformes":"charadriiformes/shorebirds", # figatebirds, boobies, gannets, cormorants
                               "unknown":"other" # I don't know
                              }

# Now my categories are anseriformes, charadriiformes and other shorebirds, accipitriformes and strigiformes, and other

strain_list_df["corrected_order"] = strain_list_df["original_order"].map(host_order_corrections_dict)

print(strain_list_df["corrected_order"].value_counts())

corrected_order
anseriformes                    2602
accipitriformes/strigiformes    1905
other                            498
charadriiformes/shorebirds       493
Name: count, dtype: int64


In [6]:
# Print the new dataframe

# strain_list_df.to_csv("./BEAST/discrete_trait_analysis/submission_files/2025-08-05_order_emptree-contphyl/inputs/taxa_order.tsv", sep = "\t")
strain_list_df.to_csv("./BEAST/continuous_phylogeography/submission_files/2025-08-11_nov2021-2024_no-suliformes_ha_contphyl_flyway-order-DTA_thorney/inputs/taxa_order.txt", sep = "\t")